# Notebook 3 — Feature Engineering
### Crash Risk Intelligence System
**Goal:** Create new meaningful features from existing columns
to improve ML model performance in Notebook 4

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

print("Libraries imported successfully")

Libraries imported successfully


In [4]:
# Load dataset saved from Notebook 2
df = pd.read_csv('C:\crash_risk_intelligence\data\eda_crash_data.csv', low_memory=False)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"\nExisting columns:")
print(df.columns.tolist())

Rows: 215,034
Columns: 37

Existing columns:
['Report Number', 'Local Case Number', 'Agency Name', 'ACRS Report Type', 'Crash Date/Time', 'Route Type', 'Road Name', 'Cross-Street Name', 'Collision Type', 'Weather', 'Surface Condition', 'Light', 'Traffic Control', 'Driver Substance Abuse', 'Person ID', 'Driver At Fault', 'Injury Severity', 'Driver Distracted By', 'Drivers License State', 'Vehicle ID', 'Vehicle Damage Extent', 'Vehicle First Impact Location', 'Vehicle Body Type', 'Vehicle Movement', 'Vehicle Going Dir', 'Speed Limit', 'Driverless Vehicle', 'Parked Vehicle', 'Vehicle Year', 'Vehicle Make', 'Vehicle Model', 'Latitude', 'Longitude', 'Location', 'Crash Hour', 'Crash Day', 'Crash Month']


## Step 1 — Time Based Features
Extracting more time features from Crash Date/Time column.
These help the model understand temporal patterns in crashes.

In [5]:
# Convert to datetime first
df['Crash Date/Time'] = pd.to_datetime(df['Crash Date/Time'], errors='coerce')

# Extract time features
df['Crash Year'] = df['Crash Date/Time'].dt.year
df['Crash Week'] = df['Crash Date/Time'].dt.isocalendar().week.astype(int)

# Is it a weekend?
df['Is Weekend'] = df['Crash Day'].isin(['Saturday', 'Sunday']).astype(int)

# Is it rush hour? (7-9am or 3-6pm)
df['Is Rush Hour'] = df['Crash Hour'].isin([7, 8, 9, 15, 16, 17, 18]).astype(int)

# Is it nighttime? (9pm to 5am)
df['Is Night'] = df['Crash Hour'].isin([21, 22, 23, 0, 1, 2, 3, 4, 5]).astype(int)

print("New time features created:")
print(f"Is Weekend — 1s: {df['Is Weekend'].sum():,}")
print(f"Is Rush Hour — 1s: {df['Is Rush Hour'].sum():,}")
print(f"Is Night — 1s: {df['Is Night'].sum():,}")

New time features created:
Is Weekend — 1s: 49,425
Is Rush Hour — 1s: 100,590
Is Night — 1s: 31,618


## Step 2 — Risk Based Features
Creating binary flags for high risk conditions identified during EDA.
These directly encode our EDA findings into model-ready features.

In [6]:
# High risk weather conditions identified in EDA
high_risk_weather = ['Blowing Sand, Soil, Dirt', 'Rain', 'Fog, Smog, Smoke', 'Snow']
df['Is High Risk Weather'] = df['Weather'].isin(high_risk_weather).astype(int)

# High risk light conditions identified in EDA
high_risk_light = ['Dark - No Lights', 'Dark - Unknown Lighting', 'Dawn', 'Dusk']
df['Is High Risk Light'] = df['Light'].isin(high_risk_light).astype(int)

# High risk speed — above 50mph speed limit
df['Is High Speed'] = (df['Speed Limit'] > 50).astype(int)

print("Risk features created:")
print(f"Is High Risk Weather — 1s: {df['Is High Risk Weather'].sum():,}")
print(f"Is High Risk Light — 1s: {df['Is High Risk Light'].sum():,}")
print(f"Is High Speed — 1s: {df['Is High Speed'].sum():,}")

Risk features created:
Is High Risk Weather — 1s: 6,229
Is High Risk Light — 1s: 8,687
Is High Speed — 1s: 4,927


## Step 3 — Driver Behavior Features
Creating features that capture driver behavior and fault patterns.
These are strong predictors for the ML model.

In [7]:
# Driver at fault binary flag
df['Is Driver At Fault'] = (df['Driver At Fault'] == 'Yes').astype(int)

# Driver substance abuse flag
df['Is Substance Abuse'] = (df['Driver Substance Abuse'] != 'None Detected').astype(int)

# Driver distracted flag
df['Is Distracted'] = (df['Driver Distracted By'] != 'Not Distracted').astype(int)

print("Driver behavior features created:")
print(f"Is Driver At Fault — 1s: {df['Is Driver At Fault'].sum():,}")
print(f"Is Substance Abuse — 1s: {df['Is Substance Abuse'].sum():,}")
print(f"Is Distracted — 1s: {df['Is Distracted'].sum():,}")

Driver behavior features created:
Is Driver At Fault — 1s: 107,081
Is Substance Abuse — 1s: 92,490
Is Distracted — 1s: 79,971


## Step 4 — Vehicle Features
Creating features that capture vehicle condition and type patterns.
These help the model understand vehicle related crash factors.

In [8]:
# Vehicle age feature
current_year = 2024
df['Vehicle Age'] = current_year - df['Vehicle Year']

# Old vehicle flag — older than 15 years
df['Is Old Vehicle'] = (df['Vehicle Age'] > 15).astype(int)

# Vehicle damage severity binary — disabling damage is most severe
df['Is Severe Damage'] = (df['Vehicle Damage Extent'] == 'Disabling').astype(int)

# Parked vehicle flag
df['Is Parked'] = (df['Parked Vehicle'] == 'Yes').astype(int)

print("Vehicle features created:")
print(f"Average vehicle age: {df['Vehicle Age'].mean():.1f} years")
print(f"Is Old Vehicle — 1s: {df['Is Old Vehicle'].sum():,}")
print(f"Is Severe Damage — 1s: {df['Is Severe Damage'].sum():,}")
print(f"Is Parked — 1s: {df['Is Parked'].sum():,}")

Vehicle features created:
Average vehicle age: 59.4 years
Is Old Vehicle — 1s: 72,104
Is Severe Damage — 1s: 80,263
Is Parked — 1s: 3,622


In [9]:
# Check Vehicle Year distribution
print("Vehicle Year stats:")
print(df['Vehicle Year'].describe())
print(f"\nVehicle Year value counts (bottom 10):")
print(df['Vehicle Year'].value_counts().sort_index().head(10))
print(f"\nUnusual years (before 1950 or after 2024):")
print(df[(df['Vehicle Year'] < 1950) | (df['Vehicle Year'] > 2024)]['Vehicle Year'].value_counts())

Vehicle Year stats:
count    215034.000000
mean       1964.618897
std         343.332337
min           0.000000
25%        2006.000000
50%        2012.000000
75%        2016.000000
max        9999.000000
Name: Vehicle Year, dtype: float64

Vehicle Year value counts (bottom 10):
Vehicle Year
0     5244
1        2
2        1
3        2
4        1
8        1
13       2
14       1
15       3
97       1
Name: count, dtype: int64

Unusual years (before 1950 or after 2024):
Vehicle Year
0       5244
2025    1098
2026     202
1900      78
9999      65
        ... 
1012       1
2205       1
9000       1
2222       1
3012       1
Name: count, Length: 80, dtype: int64


In [10]:
# Replace invalid years with NaN
df['Vehicle Year'] = df['Vehicle Year'].replace(0, np.nan)
df.loc[df['Vehicle Year'] > 2024, 'Vehicle Year'] = np.nan
df.loc[df['Vehicle Year'] < 1950, 'Vehicle Year'] = np.nan

# Fill nulls with median year
median_year = df['Vehicle Year'].median()
df['Vehicle Year'] = df['Vehicle Year'].fillna(median_year)

# Now calculate vehicle age correctly
current_year = 2024
df['Vehicle Age'] = current_year - df['Vehicle Year']

# Old vehicle flag
df['Is Old Vehicle'] = (df['Vehicle Age'] > 15).astype(int)

print("Vehicle Year stats after cleaning:")
print(df['Vehicle Year'].describe())
print(f"\nAverage vehicle age: {df['Vehicle Age'].mean():.1f} years")
print(f"Is Old Vehicle — 1s: {df['Is Old Vehicle'].sum():,}")

Vehicle Year stats after cleaning:
count    215034.000000
mean       2011.534771
std           6.492367
min        1955.000000
25%        2007.000000
50%        2013.000000
75%        2016.000000
max        2024.000000
Name: Vehicle Year, dtype: float64

Average vehicle age: 12.5 years
Is Old Vehicle — 1s: 66,696
